# 📚 Fiche de Révision Ultime : Machine Learning IMDS

Cette fiche rassemble les fonctions clés de tes TPs (implémentées from scratch avec `numpy`) et les outils de base (`pandas`, `scipy`) utiles pour ton examen sur machine.

## 1. Importations et Préparation des Données
Lors de l'examen, la première étape est de charger et de formater correctement les données. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat
from scipy import optimize

# Uniquement si besoin de traiter des catégories ou mélanger :
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

### Charger les données et Gérer les catégories

In [2]:
### --- POUR UN FICHIER CSV ---
# df = pd.read_csv('data.csv')
# Transformation des catégories (ex: "Male"/"Female") en nombres (0, 1)
# le = LabelEncoder()
# df['gender'] = le.fit_transform(df['gender'])

# Mélanger les données (très important avant de découper)
# data = np.array(df)
# data = shuffle(data)

# Séparation X et y (adapter les indices selon les colonnes de l'examen)
# X = data[:, 0:-1] # Toutes les colonnes sauf la dernière
# y = data[:, -1]   # La dernière colonne

### --- POUR UN FICHIER MAT ---
# data = loadmat('data.mat')
# X = data['X']
# y = data['y'].flatten() # Aplatir en vecteur 1D

### Fractionner en Train / Validation / Test et Ajouter le biais

In [3]:
def split_data(X, y, train_ratio=0.6, val_ratio=0.2):
    m = X.shape[0]
    split_1 = int(train_ratio * m)
    split_2 = int((train_ratio + val_ratio) * m)
    
    X_train, y_train = X[:split_1, :], y[:split_1]
    X_val, y_val = X[split_1:split_2, :], y[split_1:split_2]
    X_test, y_test = X[split_2:, :], y[split_2:]
    
    return X_train, y_train, X_val, y_val, X_test, y_test

def add_bias(X):
    m = X.shape[0]
    return np.concatenate([np.ones((m, 1)), X], axis=1)

## 2. Le Secret des Dimensions (Questions d'Examen)
- **$X$ (Matrice des caractéristiques) :** Dimension $(m, n)$ où $m$ = exemples, $n$ = features. Avec biais : $(m, n+1)$.
- **$y$ (Vecteur cible) :** Dimension $(m, 1)$ ou $(m,)$.
- **$\theta$ (Vecteur des paramètres) :** Dimension $(n+1, 1)$ ou $(n+1,)$.

**Produit matriciel (`np.dot`) :** 
Le nombre de *colonnes* de la première matrice doit égaler le nombre de *lignes* de la deuxième.
- Hypothèse : `h = np.dot(X, theta)` -> $(m, n+1) \times (n+1, 1) = (m, 1)$
- Gradient : `grad = np.dot(X.T, error)` -> $(n+1, m) \times (m, 1) = (n+1, 1)$

## 3. Régression Linéaire (avec Régularisation)
**Formules :**
* Coût : $J(\theta) = \frac{1}{2m} \sum (h_\theta(x) - y)^2 + \frac{\lambda}{2m} \sum_{j=1}^n \theta_j^2$
* Gradients : $\frac{\partial J}{\partial \theta} = \frac{1}{m} X^T (h_\theta(x) - y) + \frac{\lambda}{m} \theta_{j \ge 1}$

In [4]:
def linearRegCostFunction(X, y, theta, lambda_=0.0):
    m = y.size
    h = np.dot(X, theta)
    
    J = (1.0 / (2 * m)) * np.sum(np.square(h - y)) + (lambda_ / (2 * m)) * np.sum(np.square(theta[1:]))
    
    grad = (1.0 / m) * np.dot(X.T, (h - y))
    grad[1:] = grad[1:] + (lambda_ / m) * theta[1:]
    
    return J, grad

def trainLinearReg(X, y, lambda_=0.0, maxiter=200):
    initial_theta = np.zeros(X.shape[1])
    costFunction = lambda t: linearRegCostFunction(X, y, t, lambda_)
    options = {'maxfun': maxiter}
    
    res = optimize.minimize(costFunction, initial_theta, jac=True, method='TNC', options=options)
    return res.x

## 4. Régression Logistique & Multi-classe
**Formules :**
* Sigmoïde : $g(z) = \frac{1}{1 + e^{-z}}$
* Coût : $J(\theta) = \frac{1}{m} \sum [-y \log(h_\theta(x)) - (1 - y) \log(1 - h_\theta(x))] + \text{Régularisation}$

In [5]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def lrCostFunction(theta, X, y, lambda_):
    m = y.size
    h = sigmoid(np.dot(X, theta))
    epsilon = 1e-5
    
    J = (1.0 / m) * np.sum(-y * np.log(h + epsilon) - (1 - y) * np.log(1 - h + epsilon)) 
    J += (lambda_ / (2 * m)) * np.sum(np.square(theta[1:]))
    
    grad = (1.0 / m) * np.dot(X.T, (h - y))
    grad[1:] = grad[1:] + (lambda_ / m) * theta[1:]
    
    return J, grad

def oneVsAll(X, y, num_labels, lambda_):
    m, n = X.shape
    all_theta = np.zeros((num_labels, n + 1))
    X_aug = np.concatenate([np.ones((m, 1)), X], axis=1)
    
    for c in range(num_labels):
        initial_theta = np.zeros(n + 1)
        options = {'maxiter': 50}
        res = optimize.minimize(lrCostFunction, initial_theta, (X_aug, (y == c), lambda_), jac=True, method='CG', options=options)
        all_theta[c, :] = res.x
        
    return all_theta

## 5. Réseaux de Neurones (Neural Networks)
Implémentation typique TP4/TP5 : `Input (A1) -> Hidden Layer (A2) -> Output (A3)`.

In [6]:
def sigma_prim(z):
    s = sigmoid(z)
    return s * (1 - s)

def compute_grad(x, y, W2, W3, b2, b3):
    m_actuel = y.shape[1] # Si y est orienté (1, m)
    
    # FORWARD
    A1 = x
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    Z3 = np.dot(W3, A2) + b3
    A3 = Z3 # Sortie sans activation pour de la regression, adapter si classification !
    
    # BACKWARD
    delta_3 = A3 - y
    delta_2 = np.dot(W3.T, delta_3) * sigma_prim(Z2)
    
    dC_W2 = (1.0 / m_actuel) * np.dot(delta_2, A1.T)
    dC_W3 = (1.0 / m_actuel) * np.dot(delta_3, A2.T)
    
    dC_b2 = (1.0 / m_actuel) * np.sum(delta_2, axis=1, keepdims=True)
    dC_b3 = (1.0 / m_actuel) * np.sum(delta_3, axis=1, keepdims=True)
    
    return dC_W2, dC_W3, dC_b2, dC_b3

## 6. Courbes d'Apprentissage (Biais vs Variance)
* **High Bias (Underfitting)** : Erreur Train et Val élevées et proches.
* **High Variance (Overfitting)** : Erreur Train très basse, erreur Val très haute (grand écart).

In [7]:
def learningCurve(X, y, Xval, yval, lambda_=0):
    m = y.size
    error_train = np.zeros(m)
    error_val = np.zeros(m)

    for i in range(1, m + 1):
        X_subset = X[:i, :]
        y_subset = y[:i]
        
        theta = trainLinearReg(X_subset, y_subset, lambda_)
        
        # Erreur calculée SANS régularisation (lambda=0)
        error_train[i-1], _ = linearRegCostFunction(X_subset, y_subset, theta, lambda_=0)
        error_val[i-1], _ = linearRegCostFunction(Xval, yval, theta, lambda_=0)
        
    return error_train, error_val

def plot_learning_curve(error_train, error_val, m):
    plt.plot(np.arange(1, m+1), error_train, label='Train Error')
    plt.plot(np.arange(1, m+1), error_val, label='Validation Error')
    plt.xlabel('Number of training examples')
    plt.ylabel('Error')
    plt.legend()
    plt.title('Learning Curve')
    plt.show()